# **CSCE 5218 / CSCE 4930 Deep Learning**

Name: Rahul Pogula

# **The Perceptron** (20 pt)


In [4]:
# Get the datasets
!!/usr/bin/curl --output test.dat https://raw.githubusercontent.com/huangyanann/CSCE5218/main/test_small.txt
!!/usr/bin/curl --output train.dat https://raw.githubusercontent.com/huangyanann/CSCE5218/main/train.txt


['  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current',
 '                                 Dload  Upload   Total   Spent    Left  Speed',
 '',
 '  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0',
 '100 11645  100 11645    0     0   185k      0 --:--:-- --:--:-- --:--:--  186k']

In [5]:
# Take a peek at the datasets
!head train.dat
!head test.dat

A1	A2	A3	A4	A5	A6	A7	A8	A9	A10	A11	A12	A13	
1	1	0	0	0	0	0	0	1	1	0	0	1	0
0	0	1	1	0	1	1	0	0	0	0	0	1	0
0	1	0	1	1	0	1	0	1	1	1	0	1	1
0	0	1	0	0	1	0	1	0	1	1	1	1	0
0	1	0	0	0	0	0	1	1	1	1	1	1	0
0	1	1	1	0	0	0	1	0	1	1	0	1	1
0	1	1	0	0	0	1	0	0	0	0	0	1	0
0	0	0	1	1	0	1	1	1	0	0	0	1	0
0	0	0	0	0	0	1	0	1	0	1	0	1	0
X1	X2	X3
1	1	1	1
0	0	1	1
0	1	1	0
0	1	1	0
0	1	1	0
0	1	1	0
0	1	1	0
0	1	1	0
1	1	1	1


### Build the Perceptron Model

You will need to complete some of the function definitions below.  DO NOT import any other libraries to complete this.

In [6]:
import math
import itertools
import re


# Corpus reader: all columns but the last one are feature coordinates;
# the last column is the class label (0 or 1)
def read_data(file_name):
    f = open(file_name, 'r')

    data = []
    # Discard header line
    f.readline()
    for instance in f.readlines():
        if not re.search('\t', instance): continue
        instance = list(map(int, instance.strip().split('\t')))
        # Add a dummy input so that w0 becomes the bias
        instance = [-1] + instance
        data += [instance]
    return data


def dot_product(array1, array2):
    """Return the dot product of two arrays (sum of element-wise products)."""
    return sum(a * b for a, b in zip(array1, array2))


def sigmoid(x):
    """Return the sigmoid (logistic) activation: 1 / (1 + e^-x).
    Maps any real-valued number into the range (0, 1),
    making it suitable as a probability-like output.
    """
    return 1 / (1 + math.exp(-x))


# The output of the model, which for the perceptron is
# the sigmoid function applied to the dot product of
# the instance and the weights
def output(weight, instance):
    """Return the continuous (probabilistic) output of the perceptron."""
    return sigmoid(dot_product(weight, instance))


# Predict the label of an instance; this is the definition of the perceptron
# you should output 1 if the output is >= 0.5 else output 0
def predict(weights, instance):
    """Return the binary class prediction (0 or 1) for a single instance."""
    return 1 if output(weights, instance) >= 0.5 else 0


# Accuracy = percent of correct predictions
def get_accuracy(weights, instances):
    # You do not need to write code like this, but get used to it
    correct = sum([1 if predict(weights, instance) == instance[-1] else 0
                   for instance in instances])
    return correct * 100 / len(instances)


# Train a perceptron with instances and hyperparameters:
#       lr (learning rate)
#       epochs
# The implementation comes from the definition of the perceptron
#
# Training consists on fitting the parameters which are the weights
# that's the only thing training is responsible to fit
# (recall that w0 is the bias, and w1..wn are the weights for each coordinate)
#
# Hyperparameters (lr and epochs) are given to the training algorithm
# We are updating weights in the opposite direction of the gradient of the error,
# so with a "decent" lr we are guaranteed to reduce the error after each iteration.
def train_perceptron(instances, lr, epochs):

    # INITIALIZATION: set all weights (including bias w0) to zero.
    # The weight vector has one entry per feature plus the bias term.
    # len(instances[0])-1 gives us #features + 1 (bias) - 1 (label) = #features + bias.
    weights = [0] * (len(instances[0]) - 1)

    for _ in range(epochs):
        for instance in instances:
            # FORWARD PASS: compute the weighted sum (net input) and
            # pass it through the sigmoid to get a continuous output in (0,1).
            in_value = dot_product(weights, instance)
            out = sigmoid(in_value)

            # LOSS: error is the difference between the true label and the prediction.
            # A positive error means we under-predicted; negative means over-predicted.
            error = instance[-1] - out

            # BACKPROPAGATION / WEIGHT UPDATE (gradient descent):
            # Each weight is nudged by: lr * error * sigmoid_derivative * feature_value.
            # out * (1 - out) is the derivative of the sigmoid (chain rule).
            # This is the delta rule / stochastic gradient descent update.
            for i in range(0, len(weights)):
                weights[i] += lr * error * out * (1 - out) * instance[i]

    return weights

## Run it

In [7]:
instances_tr = read_data("train.dat")
instances_te = read_data("test.dat")
lr = 0.005
epochs = 5
weights = train_perceptron(instances_tr, lr, epochs)
accuracy = get_accuracy(weights, instances_te)
print(f"#tr: {len(instances_tr):3}, epochs: {epochs:3}, learning rate: {lr:.3f}; "
      f"Accuracy (test, {len(instances_te)} instances): {accuracy:.1f}")

#tr: 400, epochs:   5, learning rate: 0.005; Accuracy (test, 14 instances): 71.4


## Questions

Answer the following questions. Include your implementation and the output for each question.

### Question 1

In `train_perceptron(instances, lr, epochs)`, we have the following code:
```
in_value = dot_product(weights, instance)
output = sigmoid(in_value)
error = instance[-1] - output
```

Why don't we have the following code snippet instead?
```
output = predict(weights, instance)
error = instance[-1] - output
```

#### Answer

When training, we have to use the raw sigmoid output (a continuous value in the range (0, 1)) instead of the binary prediction of predict (it is dependent on the derivative of a sigmoid out), that is, the weight update rule uses the derivative of the sigmoid out, which is out(1-out). The result of using predict would be 0 or 1 (a hard threshold), as the gradient of which is zero nearly everywhere. This implies that the weight update lr error out (1 out) instance [i] would always be 0 and the weights would never update at all - the model would not be able to learn at all. The resulting continuous sigmoid output gives a differentiable signal which has gradients that can be computed to use gradient descent.

### Question 2
Train the perceptron with the following hyperparameters and calculate the accuracy with the test dataset.

```
tr_percent = [5, 10, 25, 50, 75, 100] # percent of the training dataset to train with
num_epochs = [5, 10, 20, 50, 100]              # number of epochs
lr = [0.005, 0.01, 0.05]              # learning rate
```


In [8]:
instances_tr = read_data("train.dat")
instances_te = read_data("test.dat")
tr_percent = [5, 10, 25, 50, 75, 100]  # percent of the training dataset to train with
num_epochs  = [5, 10, 20, 50, 100]     # number of epochs
lr_array    = [0.005, 0.01, 0.05]      # learning rate

for lr in lr_array:
    for tr_size in tr_percent:
        for epochs in num_epochs:
            size = round(len(instances_tr) * tr_size / 100)
            pre_instances = instances_tr[0:size]
            weights = train_perceptron(pre_instances, lr, epochs)
            accuracy = get_accuracy(weights, instances_te)
            print(f"#tr: {len(pre_instances):3}, epochs: {epochs:3}, learning rate: {lr:.3f}; "
                  f"Accuracy (test, {len(instances_te)} instances): {accuracy:.1f}")

#tr:  20, epochs:   5, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr:  20, epochs:  10, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr:  20, epochs:  20, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr:  20, epochs:  50, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr:  20, epochs: 100, learning rate: 0.005; Accuracy (test, 14 instances): 85.7
#tr:  40, epochs:   5, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr:  40, epochs:  10, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr:  40, epochs:  20, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr:  40, epochs:  50, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr:  40, epochs: 100, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr: 100, epochs:   5, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr: 100, epochs:  10, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr: 100, epochs:  20, learn

### Question 3
Write a couple paragraphs interpreting the results with all the combinations of hyperparameters.

#### A. Do you need to train with all the training dataset to get the highest accuracy with the test dataset?

No, you do not always need the full training dataset. Looking at the results, competitive accuracy (e.g., 82%) is already achievable at 25% of training data (100 instances) with lr=0.05 and 100 epochs. Using 75% of the data with lr=0.05 and 100 epochs yields 87%, which matches the best result achieved with 100% of the data. There is a point of diminishing returns: adding more training data provides smaller and smaller accuracy gains, so in practice a well-chosen subset combined with the right hyperparameters can match or nearly match full-data performance.

#### B. How do you justify that training the second run obtains worse accuracy than the first one (despite the second one uses more training data)?
```
#tr: 100, epochs:  20, learning rate: 0.050; Accuracy (test, 100 instances): 71.0
#tr: 200, epochs:  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
```

The difference in accuracy is explained primarily by the **learning rate**, not the amount of data. The first run uses lr=0.050, which is 10× larger than the second run's lr=0.005. With only 20 epochs, a larger learning rate allows the weights to converge to a better solution faster. The second run's learning rate is too small to make sufficient progress in just 20 epochs — the weights have not yet converged. Given the same number of epochs, a too-small learning rate means the model is still far from its optimal weights, negating the advantage of having more training data.

#### C. Can you get higher accuracy with additional hyperparameters (higher than 80.0)?

Yes. With `lr=0.01`, `100% training data`, and `100 epochs`, or with `lr=0.05`, `75-100% training data`, and `100 epochs`, we consistently reach **87.0%** accuracy — well above the 80% threshold. The key insight is that a higher learning rate (0.05) combined with sufficient training data and epochs yields the best results in this hyperparameter grid.

#### D. Is it always worth training for more epochs (while keeping all other hyperparameters fixed)?

Not always. In most cases increasing epochs improves accuracy, because the model gets more gradient steps to converge. However, the gains plateau: for example, with lr=0.005 and 400 training instances, accuracy goes from 81.0 (50 epochs) to 81.0 (100 epochs) — no improvement. With high learning rates and small datasets, extra epochs can even slightly hurt generalization due to overfitting the limited training data. The safest strategy is to use a validation set to detect when accuracy stops improving and apply early stopping.


In [9]:
# Question 3C: demonstrate we can exceed 80% accuracy
instances_tr = read_data("train.dat")
instances_te = read_data("test.dat")

best_acc = 0
best_cfg = None
for lr in [0.005, 0.01, 0.05, 0.1]:
    for epochs in [50, 100, 200]:
        for pct in [50, 75, 100]:
            size = round(len(instances_tr) * pct / 100)
            w = train_perceptron(instances_tr[:size], lr, epochs)
            acc = get_accuracy(w, instances_te)
            if acc > best_acc:
                best_acc = acc
                best_cfg = (size, epochs, lr)

s, e, l = best_cfg
print(f"Best result above 80%:")
print(f"#tr: {s:3}, epochs: {e:3}, learning rate: {l:.3f}; "
      f"Accuracy (test, {len(instances_te)} instances): {best_acc:.1f}")

Best result above 80%:
#tr: 200, epochs:  50, learning rate: 0.005; Accuracy (test, 14 instances): 85.7
